# Missing data: which columns disappear together?

**Two tables, no charts:** first see how much is missing, then see the dates and lengths of shared gaps. No data is filled or deleted. Put `dataset_mansion.csv` beside this notebook, or change the path below.

In [1]:
from pathlib import Path
import pandas as pd

path = Path('dataset_mansion.csv')
if not path.exists():
    path = Path('upload/dataset_mansion.csv')

df = pd.read_csv(path, index_col=0)
df['Consumption'] = pd.to_numeric(
    df['Consumption'].astype('string').str.replace(',', '.', regex=False),
    errors='raise'
)
df['MTime'] = pd.to_datetime(df['MTime'], errors='raise')
assert df['MTime'].notna().all(), 'Check missing timestamps first.'
assert not df['MTime'].duplicated().any(), 'Check duplicate timestamps first.'
df = df.set_index('MTime').sort_index()
expected = pd.date_range(df.index.min(), df.index.max(), freq='h')
assert df.index.equals(expected), 'This version expects a complete hourly timeline.'
missing = df.isna()

### 1. Which columns have missing values?

Percentages use all **56,231 hourly observations**. The timeline is complete; these are missing values inside existing rows.

In [2]:
summary = pd.DataFrame({
    'Missing hours': missing.sum(),
    'Missing (%)': missing.mean().mul(100).round(2)
}).sort_values('Missing hours', ascending=False)
display(summary)

,Missing hours,Missing (%)
PresentWeather(rank),32441,57.69
MaximumWindSpeed(m/s),741,1.32
MinimumWindSpeed(m/s),741,1.32
WindSpeed(m/s),618,1.10
WindDirection(deg),501,0.89
MaximumPrecipitationIntensity(mm/h),321,0.57
PrecipitationAmount(mm),303,0.54
RelativeHumidity(%),289,0.51
AirPressure(hPa),210,0.37
AirTemperature(degC),208,0.37


### 2. When do multiple columns go missing together?

Each row below is a **continuous run of hours with the same set of missing columns**. If that set changes, a new event starts. Only events affecting at least two columns are included.

`PresentWeather(rank)` is left out of this shared-event calculation because its long missing periods would obscure the other gaps. It remains in Table 1 and is discussed below. “Weather measurements” below means the 11 weather columns other than weather rank.

The table shows the **10 longest shared events**. Start and end are both included: 10:00 through 05:00 the next day means 20 missing hourly observations. Timestamps follow the CSV; no timezone is specified.

In [3]:
# Leave weather rank aside when identifying shared gaps.
checked_cols = [c for c in df.columns if c != 'PresentWeather(rank)']
mask = missing[checked_cols]

# A new event begins whenever any column changes missing/observed status.
new_event = mask.ne(mask.shift()).any(axis=1)
event_id = new_event.cumsum()

location_cols = ['Latitude', 'Longitude', 'Place']
weather_cols = [c for c in checked_cols
                if c not in ['Consumption'] + location_cols]

def describe_columns(columns):
    # Shorten the label only when the entire group is missing.
    remaining = list(columns)
    labels = []
    wind_cols = ['WindSpeed(m/s)', 'MaximumWindSpeed(m/s)',
                 'MinimumWindSpeed(m/s)', 'WindDirection(deg)']
    rain_cols = ['PrecipitationAmount(mm)', 'MaximumPrecipitationIntensity(mm/h)']
    for group, label in [(location_cols, 'All 3 location columns'),
                         (weather_cols, 'All 11 weather measurements'),
                         (wind_cols, 'All 4 wind columns'),
                         (rain_cols, 'Both precipitation columns')]:
        if set(group).issubset(remaining):
            labels.append(label)
            remaining = [c for c in remaining if c not in group]
    return '; '.join(labels + remaining)

rows = []
for _, block in mask.groupby(event_id):
    absent = block.columns[block.iloc[0]].tolist()
    if len(absent) < 2:
        continue
    rows.append({
        'Start': block.index[0],
        'End': block.index[-1],
        'Missing hours': len(block),
        'Missing together (excluding weather rank)': describe_columns(absent),
        'Consumption available throughout?':
            'Yes' if not missing.loc[block.index, 'Consumption'].any() else 'No'
    })

# All events remain available in this variable for further inspection.
events = pd.DataFrame(rows).sort_values(
    ['Missing hours', 'Start'], ascending=[False, True]
).reset_index(drop=True)

display(events.head(10))
# To inspect more events, change 10 above, or use display(events).

,Start,End,Missing hours,Missing together (excluding weather rank),Consumption available throughout?
0,2017-05-14 06:00:00,2017-05-16 15:00:00,58,All 4 wind columns,Yes
1,2017-05-11 15:00:00,2017-05-13 00:00:00,34,All 4 wind columns,Yes
2,2021-07-18 10:00:00,2021-07-19 05:00:00,20,All 3 location columns; All 11 weather measurements,Yes
3,2016-01-18 07:00:00,2016-01-18 18:00:00,12,Both precipitation columns,Yes
4,2016-01-19 19:00:00,2016-01-20 06:00:00,12,Both precipitation columns,Yes
5,2016-01-20 19:00:00,2016-01-21 06:00:00,12,Both precipitation columns,Yes
6,2016-05-20 07:00:00,2016-05-20 18:00:00,12,Both precipitation columns,Yes
7,2017-05-21 12:00:00,2017-05-21 23:00:00,12,All 4 wind columns,Yes
8,2017-05-13 10:00:00,2017-05-13 19:00:00,10,All 4 wind columns,Yes
9,2017-05-19 14:00:00,2017-05-19 23:00:00,10,All 4 wind columns,Yes


### Three findings to take away

1. **Location and weather share gaps.** Latitude, longitude and place are missing at exactly the same **191 hours**, and all weather columns are also missing at those hours. Those location gaps form **151 continuous runs**, usually one hour long. Their longest shared gap lasts **20 hours**, from **18 July 2021 at 10:00** through **19 July at 05:00**. Consumption remains available.
2. **Weather rank has a separate coverage pattern.** It is missing from the start through **24 September 2018 at 09:00** (23,940 hours), and from **18 June 2021 at 07:00** through the end (8,342 hours), plus some shorter gaps. Do not interpret a missing rank as zero or clear weather without its source definition.
3. **There is clear structure in the missingness.** Some columns disappear together, and some gaps occur in long blocks. Other gaps are scattered. These observations do not prove statistical randomness or identify the cause.

**How to use the event table:** start with the longest gaps, inspect the affected columns, and check whether these dates match known source outages or data-merging problems. The CSV alone cannot establish the reason. Single-column gaps are outside this shared-event table, and a column's total outage may span several rows if other columns change status during it.